# AI-Driven Behavior-Based Intrusion Detection System
**Author:** Justus Izuchukwu Onuh | **Dataset:** CSIC 2010 HTTP Dataset



## Introduction: The Paradigm Shift in Web Security

### The Problem: Why Traditional Firewalls Fail
For decades, web security has relied on **Rule-Based Firewalls**. These systems act like a security guard checking a strict list of known threats (signatures). If a hacker uses a known attack, the firewall blocks it. However, if a hacker uses a brand new, slightly modified attack (a "Zero-Day" exploit), the traditional firewall doesn't recognize it and lets it right through. Hackers are evolving faster than we can write new rules.

### The Solution: Behavior-Based AI
Instead of memorizing a list of known attacks, this project utilizes a **Deep Neural Network (DNN)** to learn *behavior*. By training the AI on the CSIC 2010 dataset—which contains thousands of normal web requests and thousands of malicious ones (like SQL Injections and Cross-Site Scripting)—the neural network learns the underlying, complex patterns of malicious intent. 

### Real-World Relevance & Project Goal
In the real world, an undetected web attack can cost a company millions in stolen data or downtime. The goal of this project is not just to train a theoretical model, but to build a **Live Intrusion Detection System (IDS)**. 

We aim to:
1. Engineer raw network traffic into a format an AI can understand.
2. Train a DNN to accurately separate safe traffic from anomalous, dangerous traffic.
3. Export this "brain" to power a real-time, Next.js web dashboard that security analysts can use to monitor live threats.

## 1. Setup and Library Imports
Importing necessary libraries including Pandas for data manipulation, Scikit-Learn for preprocessing, Imbalanced-Learn for SMOTE, and TensorFlow/Keras for the Deep Neural Network.

In [6]:

# 1. Setup and Library Imports


# I am importing Pandas and Numpy for core data manipulation and matrix operations.
import pandas as pd
import numpy as np

# For visualizing the class distributions and evaluation metrics (like the confusion matrix) later.
import matplotlib.pyplot as plt
import seaborn as sns

# I need Scikit-Learn for preprocessing the raw data and evaluating the final model's performance.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score

# Importing SMOTE to handle the severe class imbalance I identified in the CSIC 2010 dataset.

from imblearn.over_sampling import SMOTE

# Setting up TensorFlow and Keras to build the Deep Neural Network (DNN) architecture.
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Setting a random seed ensures that my data splits and neural network weight initializations 
# are reproducible every time I run this notebook.
np.random.seed(42)
tf.random.set_seed(42)

print("Environment setup complete. All necessary libraries successfully imported.")

Environment setup complete. All necessary libraries successfully imported.


## 2. Data Acquisition & Parsing
The CSIC 2010 dataset comes as raw HTTP requests. Here, we parse the raw text logs and extract structural features (e.g., payload length, method type, URL anomalies) into a structured Pandas DataFrame.

In [5]:
# ==========================================
# 2. Production Data Acquisition & Parsing
# ==========================================
import os
import urllib.request
import urllib.parse
import re
import pandas as pd
import numpy as np

print("Initializing production data acquisition pipeline...")

# --- Step 1: Directory Setup ---
data_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(data_dir, exist_ok=True) 

normal_file = os.path.join(data_dir, 'normalTrafficTraining.txt')
anomalous_file = os.path.join(data_dir, 'anomalousTrafficTest.txt')

# Using a verified active mirror for the CSIC 2010 dataset
urls = {
    normal_file: "https://raw.githubusercontent.com/Monkey-D-Groot/Machine-Learning-on-CSIC-2010/master/normalTrafficTraining.txt",
    anomalous_file: "https://raw.githubusercontent.com/Monkey-D-Groot/Machine-Learning-on-CSIC-2010/master/anomalousTrafficTest.txt"
}

# --- Download Protocol ---
for filepath, url in urls.items():
    if not os.path.exists(filepath):
        print(f"Downloading real dataset: {os.path.basename(filepath)}...")
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as response, open(filepath, 'wb') as out_file:
                out_file.write(response.read())
            print(f"Successfully downloaded: {os.path.basename(filepath)}")
        except Exception as e:
            raise RuntimeError(f"FATAL ERROR: Could not download {os.path.basename(filepath)}. {e}")
    else:
        print(f"Found existing local dataset: {os.path.basename(filepath)}")

# --- Step 2: Feature Extraction Logic ---
def extract_features_from_request(request_string, is_anomalous=0):
    features = {}
    features['request_length'] = len(request_string)
    features['method_POST'] = 1 if request_string.startswith('POST') else 0
    features['method_GET'] = 1 if request_string.startswith('GET') else 0
    features['method_PUT'] = 1 if request_string.startswith('PUT') else 0
    
    decoded_request = urllib.parse.unquote(request_string).lower()
    features['special_char_count'] = len(re.findall(r"[<>\'\";\(\)\=]", decoded_request))
    features['sqli_keyword_count'] = sum(decoded_request.count(kw) for kw in ['union', 'select', 'insert', 'drop', 'or 1=1'])
    features['xss_keyword_count'] = sum(decoded_request.count(kw) for kw in ['script', 'alert', 'onerror', 'onload'])
    
    features['label'] = is_anomalous
    return features

def parse_csic_file(filepath, is_anomalous):
    print(f"Executing deep extraction on: {os.path.basename(filepath)}...")
    parsed_data = []
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as file:
        raw_requests = file.read().split('\n\n')
        for req in raw_requests:
            if len(req.strip()) > 5:
                parsed_data.append(extract_features_from_request(req, is_anomalous))
    return pd.DataFrame(parsed_data)

# --- Step 3: Execution ---
# We no longer fall back. We only use the real data.
df_normal = parse_csic_file(normal_file, is_anomalous=0)
df_anomalous = parse_csic_file(anomalous_file, is_anomalous=1)

df = pd.concat([df_normal, df_anomalous], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nPipeline complete. Total REAL requests vectorized: {len(df)}")
print(f"Real Class Distribution:\n{df['label'].value_counts(normalize=True) * 100}")

df.head()

Initializing production data acquisition pipeline...
Found existing local dataset: normalTrafficTraining.txt
Found existing local dataset: anomalousTrafficTest.txt
Executing deep extraction on: normalTrafficTraining.txt...
Executing deep extraction on: anomalousTrafficTest.txt...

Pipeline complete. Total REAL requests vectorized: 77527
Real Class Distribution:
label
0    55.464548
1    44.535452
Name: proportion, dtype: float64


,request_length,method_POST,method_GET,method_PUT,special_char_count,sqli_keyword_count,xss_keyword_count,label
0,564,0,1,0,22,0,0,0
1,500,0,0,0,17,0,0,0
2,767,0,0,0,30,0,0,0
3,571,0,0,0,17,0,0,1
4,534,0,0,0,18,0,0,0


## 3. Train / Validation / Test Split
**Crucial Step:** We must split our dataset *before* applying any synthetic oversampling to prevent data leakage. The test set must remain entirely unseen and represent real-world imbalanced traffic.

In [ ]:
code here

## 4. Feature Scaling
Applying `MinMaxScaler` to normalize our numerical features (like payload length) so they do not overpower the binary categorical flags (0/1) during the neural network's gradient descent.

In [ ]:
code here

## 5. Handling Class Imbalance (SMOTE)
Applying the Synthetic Minority Over-sampling Technique (SMOTE) strictly to the **training set** to balance the "Normal" vs. "Anomalous" traffic, preventing the model from predicting the majority class by default.

In [ ]:
code here

## 6. Deep Neural Network Architecture
Building the baseline Keras model.
- **Hidden Layers:** 3 dense layers utilizing the `ReLU` activation function.
- **Output Layer:** 1 dense layer utilizing the `Sigmoid` activation function for binary classification.
- **Regularization:** Implementing `Dropout` layers to randomly deactivate neurons and prevent the 98% overfitting issue previously encountered.

In [ ]:
code here

## 7. Model Compilation & Training
Compiling the model with binary crossentropy loss. We implement an `EarlyStopping` callback monitoring the validation loss to halt training the moment the model begins to overfit.

In [ ]:
code here

## 8. Rigorous Model Evaluation
Because raw accuracy is a flawed metric for imbalanced cybersecurity data, we evaluate the model using:
- The Confusion Matrix
- Precision (Minimizing False Positives)
- Recall
- F1-Score

In [ ]:
code here

## 9. Model Export for API Integration
Saving the trained Keras model (`.h5` format) and the fitted `MinMaxScaler` object so they can be loaded into our Python FastAPI microservice to serve the Next.js live dashboard.